
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from your .env file
load_dotenv(dotenv_path="../.env")
load_dotenv()

# Set keys in the environment for standard LangChain integrations

# Set keys in the environment only if they are not None
for key in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY", "QWEN_API_KEY"]:
    val = os.getenv(key)
    if val is not None:
        os.environ[key] = val


### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

# 2. Initialize Qwen (uses OpenAI-compatible provider, passing specific configuration parameters)
qwen_unified = init_chat_model(
    model=os.getenv("QWEN_MODEL", "qwen-turbo"),
    model_provider="openai",
    api_key=os.getenv("QWEN_API_KEY"),
    base_url=os.getenv("QWEN_BASE_URL")
)

model = qwen_unified

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000022BE47278C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022BE4870440>, root_client=<openai.OpenAI object at 0x0000022BE3E1EBA0>, root_async_client=<openai.AsyncOpenAI object at 0x0000022BE48701A0>, model_name='qwen-turbo', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://dashscope-intl.aliyuncs.com/compatible-mode/v1', openai_proxy=None, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 't

In [6]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception** (2010) is a science fiction action film directed by **Christopher Nolan**, written by Christopher Nolan and his brother **Jonathan Nolan**, and produced by **Thomas Tull** and **Emma Thomas**. It stars **Leonardo DiCaprio** as **Dominic Cobb**, a professional thief who steals information by infiltrating the subconscious of his targets. Instead of stealing, he is offered a chance to have his criminal history erased by performing the reverse: planting an idea into someone\'s mind.\n\n---\n\n## 🎬 **Plot Summary**\n\nIn a near-future world, **Cobb** is a skilled **extractor**—a thief who enters people\'s dreams to steal secrets. He is offered a chance to clear his criminal record by performing **inception**, a process that involves planting an idea into a target\'s subconscious rather than stealing it.\n\nThe target is **Robert Fischer**, the heir to a vast business empire. Cobb and his team must enter Fischer\'s dream and implant the idea that he should br

In [7]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### MEssage output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='{"director": "Christopher Nolan", "rating": 8.8, "title": "Inception", "year": 2010}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 21, 'total_tokens': 52, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-865aa6c1-76af-95ba-bca8-14fb70cc10d8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f46c9-3523-73a0-8428-8c89e798c5eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 31, 'total_tokens': 52, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dominic Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Tom Berenger', role='Malcolm'), Actor(name='Michael Caine', role='Robert Fischer'), Actor(name='Cillian Murphy', role='Robert Fischer (young)')], genres=['Action', 'Adventure', 'Science Fiction'], budget=150000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon',
 'rating': 8.0,
 'title': 'The Avengers',
 'year': 2012}

In [11]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 2.5}

In [ ]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os

# Set keys in the environment only if they are not None
for key in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY", "QWEN_API_KEY"]:
    val = os.getenv(key)
    if val is not None:
        os.environ[key] = val


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e0780144-9749-4c69-99c5-8eb8b4360de8'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 204, 'total_tokens': 304, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CngdUw4RmLEHi79ZGRNcn94b344n7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--afd570d1-986e-44ad-b4e5-8f32716216fa-0', usage_metadata={'input_tokens': 204, 'output_tokens': 100, 'total_tokens': 304, 'in

In [ ]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')